# Paper-comparable (leave-one-out) training on a GPU

Trains the two neural models of the leave-one-out track — **SASRec** (reference baseline) and the **Semantic-ID generative transformer** — and runs `run_loo_evaluation.py` on all users. See `docs/09_audit_sources_and_results.md` and `docs/10_loo_protocol.md` for why this protocol exists.

Runtime → *Change runtime type* → **GPU** (T4 is plenty). Both trainers resume from any checkpoint already in `data/processed/`, so re-running this notebook just adds epochs.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# >>> point this at the repo folder inside your Drive <<<
REPO = '/content/drive/Othercomputers/My laptop/generative-recommendation-engine'
%cd $REPO
!ls backend/scripts | head -30

Mounted at /content/drive
/content/drive/Othercomputers/My laptop/generative-recommendation-engine
build_item_embeddings_local.py
build_item_embeddings.py
build_semantic_sequences.py
build_sequences.py
diagnose_retrieval.py
download_amazon_reviews.py
evaluate_ranking.py
evaluate_retrieval.py
export_csv.py
prepare_data_amazon.py
prepare_data.py
__pycache__
run_baselines.py
run_full_evaluation.py
run_loo_evaluation.py
split_data_loo.py
split_data.py
train_ranker.py
train_rqvae.py
train_sasrec.py
train_transformer.py


In [3]:
# Colab ships JAX with CUDA support; install the rest. (implicit needs a compile on some images - ~1 min.)
!pip install -q "jax[cuda12]" implicit lightgbm pyarrow 2>&1 | tail -1
import jax; print(jax.__version__, jax.devices())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 32.8 MB/s eta 0:00:00
0.11.1 [CudaDevice(id=0)]


## 1. Data for the LOO track (fast, idempotent)
Skip if `data/processed/loo_*.parquet` and `transformer_train_sequences_loo.npz` already exist.

In [ ]:
import os
if not os.path.exists('data/processed/loo_train.parquet'):
    !python backend/scripts/split_data_loo.py
if not os.path.exists('data/processed/transformer_train_sequences_loo.npz'):
    !python backend/scripts/build_semantic_sequences.py --loo --max-items 20 --user-buckets 2000

## 2. SASRec baseline
Loss should flatten somewhere around 6.x–7.x (full-softmax over 25.6K items). 30 epochs is a reasonable budget; each epoch prints its time so you can extend.

In [ ]:
!python backend/scripts/train_sasrec.py --status
!python backend/scripts/train_sasrec.py --epochs 30 --batch-size 256

epoch: 6  config: {'d_model': 64, 'n_heads': 2, 'n_layers': 2, 'd_ff': 256, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 128, 'max_items': 20}
loss history: [9.0263, 8.3253, 8.0214, 7.8303, 7.6853, 7.5647]
resuming from epoch 6 (config: {'d_model': 64, 'n_heads': 2, 'n_layers': 2, 'd_ff': 256, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 128, 'max_items': 20})
epoch   7  loss 7.4628  (153s)
epoch   8  loss 7.3772  (149s)
epoch   9  loss 7.3008  (153s)
epoch  10  loss 7.2318  (158s)
epoch  11  loss 7.1697  (163s)
epoch  12  loss 7.1112  (171s)
epoch  13  loss 7.0596  (179s)
epoch  14  loss 7.0125  (182s)
epoch  15  loss 6.9667  (189s)
epoch  16  loss 6.9275  (193s)
epoch  17  loss 6.8886  (198s)
epoch  18  loss 6.8513  (206s)
epoch  19  loss 6.8185  (212s)
epoch  20  loss 6.7890  (219s)
epoch  21  loss 6.7584  (228s)
epoch  22  loss 6.7349  (235s)
epoch  23  loss 6.7056  (240s)
epoch  24  loss 6.6840  (251s)
epoch  25  loss 6.6596  (251s)
epoch  26  loss 6.6410  (256s)
epoch  27  loss 6.618

## 3. Semantic-ID transformer (TIGER-style: 4 layers, d=128, dropout 0.1, user token, 20-item context)
The architecture flags only matter on a fresh start; afterwards they are read from the checkpoint. Loss is per Semantic-ID *digit* (vocab 2,790), so it is not directly comparable to SASRec's.

In [ ]:
!python backend/scripts/train_transformer.py --loo --status
!python backend/scripts/train_transformer.py --loo --epochs 40 --d-model 128 --layers 4 --d-ff 512 --dropout 0.1 --batch-size 256

epoch: 5  config: {'d_model': 128, 'n_heads': 4, 'n_layers': 4, 'd_ff': 512, 'dropout': 0.1, 'lr': 0.0003, 'batch_size': 128}
loss history: [4.1617, 3.2238, 2.8753, 2.6919, 2.5847]
resuming from epoch 5 (config: {'d_model': 128, 'n_heads': 4, 'n_layers': 4, 'd_ff': 512, 'dropout': 0.1, 'lr': 0.0003, 'batch_size': 128})
epoch   6  loss 2.5143  (228s)
epoch   7  loss 2.4644  (228s)
epoch   8  loss 2.4269  (228s)
epoch   9  loss 2.3975  (237s)
epoch  10  loss 2.3726  (238s)
epoch  11  loss 2.3529  (245s)
epoch  12  loss 2.3358  (250s)
epoch  13  loss 2.3207  (252s)
epoch  14  loss 2.3077  (258s)
epoch  15  loss 2.2961  (264s)
epoch  16  loss 2.2854  (270s)
epoch  17  loss 2.2756  (277s)
epoch  18  loss 2.2674  (281s)
epoch  19  loss 2.2595  (289s)
epoch  20  loss 2.2517  (294s)
epoch  21  loss 2.2449  (300s)
epoch  22  loss 2.2390  (305s)
epoch  23  loss 2.2331  (307s)
epoch  24  loss 2.2273  (314s)
epoch  25  loss 2.2227  (319s)
epoch  26  loss 2.2176  (330s)
epoch  27  loss 2.2127  (335

## 4. Evaluate every tier on the same users (`--n-users 0` = all ~95K users; a few minutes on GPU)

In [4]:
!python backend/scripts/run_loo_evaluation.py --split both --n-users 0

/usr/local/lib/python3.13/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(
fitting popularity + ALS on LOO train...
100% 15/15 [00:10<00:00,  1.45it/s]
setup done in 24.7s

=== val: 94,762 users ===
  [sasrec] checkpoint epoch 36, loss 6.4828
  sasrec scoring: 22.7s
  [transformer] checkpoint epoch 45, loss 2.1582
    beam search 1,000/94,762 (27s)
    beam search 2,000/94,762 (49s)
    beam search 3,000/94,762 (68s)
    beam search 4,000/94,762 (87s)
    beam search 5,000/94,762 (106s)
    beam search 6,000/94,762 (125s)
    beam search 7,000/94,762 (144s)
    beam search 8,000/94,762 (163s)
    beam search 9,000/94,762 (182s)
    beam search 10,000/94,762 (205s)
    beam search 11,000/94,762 (224s)
    beam search 12,000/94,762 (244s)
    beam search 13,000/94,762 (263s)
    beam search 14,000/94,762 (282s)
    beam search 15,000/94,762 (301s)
    beam

## 5. Optional: keep training if the transformer loss is still dropping
Re-run cells 3 and 4. To compare against the paper directly, the numbers to look at are the `transformer` row vs the `sasrec` row on **this** table (same data, same protocol); TIGER reports +7–17% Recall@10 over SASRec on Amazon Beauty/Sports/Toys.